# NV Magnetometry & 3D Field Reconstruction Benchmark using Fourier

Evaluates Tikhonov–Fourier reconstruction of **B** from a single NV projection $B_\parallel = \mathbf{B} \cdot \hat{u}$.

This notebook seek to challenge the reconstruction algorithm to better understand its limitations, especially for the fourier mode $\mathbf{k}=(0,0)$.  
Four ground-truth variants stress the **k=0 mode** of the algorithm:
- **Classic**: pure dipole field from magnetic domains, with mean close to 0 (almost no mean constant field)
- **Classic + in-plane B**: Bx bias (constant field in the 2D x,y plane)
- **Classic + Bz bias**: Bz bias (constant field in the z direction)
- **Classic + NV bias**: bias along NV (constant field in 3D)

In [ ]:
%matplotlib inline
from pathlib import Path
ROOT = Path.cwd()
assert (ROOT / "helpers.py").is_file(), "Launch from the repository root."
OUT = ROOT / "outputs" / "benchmark"
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.fft import fft2, ifft2, ifftshift
from scipy.signal import convolve2d
from itertools import product as iproduct
import pandas as pd
from scipy.ndimage import gaussian_filter

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titlesize'] = 10

## Parameters

In [ ]:
#*** Grid & sample ************************************************************
NX = NY   = 110
Lx = Ly   = 3e-6           # sample size [m]
dx        = Lx / NX        # pixel size  [m]

z_meas     = 100e-9        # measurement height [m]
z_meas_idx = max(1, int(round(z_meas / dx)))
z_val      = z_meas_idx * dx

#*** Magnetic film ************************************************************
Ms     = 8e5               # saturation magnetization [A/m]
t_film = 10e-9             # film thickness [m]
C_dip  = 1e-7 * Ms * dx**2 * t_film   # dipole prefactor [T·m^3]

#*** NV axis (theta = 125.25°, phi = 90°) *************************************
theta_nv = np.deg2rad(125.25)
phi_nv   = np.deg2rad(90.0)
nv_hat   = np.array([
    np.sin(theta_nv) * np.cos(phi_nv),
    np.sin(theta_nv) * np.sin(phi_nv),
    np.cos(theta_nv)
])
ux, uy, uz = nv_hat

#*** Bias fields (amplitude chosen ~comparable to dipole signal) **************
B_bias_amp     = 10e-3 #10e-3      # [T]
B_bias_inplane = np.array([B_bias_amp,  0.,          0.         ])
B_bias_z       = np.array([0.,          0.,          B_bias_amp ])
B_bias_nv      = B_bias_amp**1 * nv_hat
print("norm B on x axis (mT) : ", np.dot(B_bias_inplane, B_bias_inplane)*1e3)
print("norm B on NV axis (mT) : ", np.dot(B_bias_nv, B_bias_nv)*1e3)

extent_um = [0, Lx*1e6, 0, Ly*1e6]

print(f"dx            = {dx*1e9:.2f} nm")
print(f"z_meas        = {z_val*1e9:.1f} nm  (index {z_meas_idx})")
print(f"NV axis       = [{ux:.4f}, {uy:.4f}, {uz:.4f}]")

## Spin lattice (ground-truth source to simulate a magnetic field)

In [ ]:
def generate_spin_lattice(nx, ny, n_domains=12, r_min=6, r_max=20, seed=4):
    """Random magnetic domains (±1) with smooth boundaries."""
    rng   = np.random.default_rng(seed)
    spins = rng.choice([-1., 1.], size=(ny, nx), p=[0.48, 0.52]) * 0.1
    for _ in range(n_domains):
        cx, cy = rng.integers(0, nx), rng.integers(0, ny)
        r      = rng.integers(r_min, r_max)
        val    = rng.choice([-1., 1.])
        yy, xx = np.ogrid[:ny, :nx]
        spins[(xx - cx)**2 + (yy - cy)**2 <= r**2] = val
    kernel = np.array([[0.5, 1., 0.5],
                       [1.,  2., 1. ],
                       [0.5, 1., 0.5]]) / 9.
    kernel /= kernel.sum()
    spins  = np.sign(convolve2d(spins, kernel, mode='same', boundary='wrap') + 0.005)
    return spins

spins = generate_spin_lattice(NX, NY)

fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(spins, origin='lower', extent=extent_um, cmap='viridis',
               vmin=-1, vmax=1, interpolation='nearest')
ax.set_title('Spin lattice (±1)'); ax.set_xlabel('x (μm)'); ax.set_ylabel('y (μm)')
plt.colorbar(im, ax=ax, label='source amplitude (dimensionless)')
plt.tight_layout(); plt.show()

In [ ]:
# Other cases to challenge the fourier reconstruction (with lower k)
def make_smooth_gradient(nx, ny, axis='x', sigma_px=8):
    xv = np.linspace(-1, 1, nx)
    yv = np.linspace(-1, 1, ny)
    if axis == 'x':
        M = np.tile(xv, (ny, 1))
    elif axis == 'y':
        M = np.tile(yv[:, None], (1, nx))
    else:
        X, Y = np.meshgrid(xv, yv, indexing='xy')
        M = (X + Y) / np.sqrt(2)
    return gaussian_filter(M, sigma=sigma_px)

def make_gaussian_blob(nx, ny, sigma_frac=0.28):
    y, x = np.mgrid[0:ny, 0:nx]
    x0, y0 = (nx - 1) / 2, (ny - 1) / 2
    sigma = sigma_frac * min(nx, ny)
    M = np.exp(-((x - x0)**2 + (y - y0)**2) / (2 * sigma**2))
    return 2 * (M - M.min()) / (M.max() - M.min()) - 1

def make_cosine_mode(nx, ny, n_periods_x=1, n_periods_y=0):
    x = np.linspace(0, 2*np.pi*n_periods_x, nx, endpoint=False)
    y = np.linspace(0, 2*np.pi*n_periods_y, ny, endpoint=False)
    X, Y = np.meshgrid(x, y, indexing='xy')
    M = np.cos(X + Y)
    return M / np.max(np.abs(M))

def make_diffuse_bubble(nx, ny, radius_frac=0.23, wall_px=6):
    y, x = np.mgrid[0:ny, 0:nx]
    x0, y0 = (nx - 1) / 2, (ny - 1) / 2
    r = np.sqrt((x - x0)**2 + (y - y0)**2)
    R = radius_frac * min(nx, ny)
    return np.tanh((R - r) / wall_px)

def make_lowk_random(nx, ny, cutoff_frac=0.06, seed=0):
    rng = np.random.default_rng(seed)
    noise = rng.normal(size=(ny, nx))
    F = np.fft.fft2(noise)
    kx = np.fft.fftfreq(nx)
    ky = np.fft.fftfreq(ny)
    KX, KY = np.meshgrid(kx, ky, indexing='xy')
    K = np.sqrt(KX**2 + KY**2)
    H = np.exp(-(K / cutoff_frac)**2)
    M = np.real(np.fft.ifft2(F * H))
    M = (M - M.mean()) / (M.std() + 1e-12)
    return np.tanh(M / 1.5)

def make_stripes(nx, ny, n_periods=2, angle_deg=0, wall_px=5):
    y, x = np.mgrid[0:ny, 0:nx]
    x = x - (nx - 1) / 2
    y = y - (ny - 1) / 2
    th = np.deg2rad(angle_deg)
    U = x*np.cos(th) + y*np.sin(th)
    period = min(nx, ny) / n_periods
    return np.tanh(np.cos(2*np.pi * U / period) / (wall_px / 3))

source_maps = {
    'Domains':              generate_spin_lattice(NX, NY),
    'Smooth gradient x':    make_smooth_gradient(NX, NY, axis='x', sigma_px=10),
    'Smooth gradient y':    make_smooth_gradient(NX, NY, axis='y', sigma_px=10),
    'Gaussian blob':        make_gaussian_blob(NX, NY, sigma_frac=0.30),
    'Cosine low-k':         make_cosine_mode(NX, NY, n_periods_x=1, n_periods_y=0),
    'Diffuse bubble':       make_diffuse_bubble(NX, NY, radius_frac=0.24, wall_px=7),
    'Low-k random':         make_lowk_random(NX, NY, cutoff_frac=0.05, seed=1),
    'Wide stripes':         make_stripes(NX, NY, n_periods=2, angle_deg=25, wall_px=7),
}


###################################################################################
# CHANGE HERE THE SOURCE MAP THAT WE WANT TO TRY FOR THE SIMULATED MAGNETIC FIELD #
spins = source_maps['Domains']  # reference case used in the manuscript
###################################################################################

fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(spins, origin='lower', extent=extent_um, cmap='viridis',
               vmin=-1, vmax=1, interpolation='nearest')
ax.set_title('Source amplitude'); ax.set_xlabel('x (μm)'); ax.set_ylabel('y (μm)')
plt.colorbar(im, ax=ax, label='source amplitude (dimensionless)')
plt.tight_layout(); plt.show()

## Dipole forward model to calculate the ground truth 3D magnetic field $B$

In [ ]:
def dipole_kernels(nx, ny, dx, z, C):
    """Real-space convolution kernels K_x, K_y, K_z for a z-polarised dipole film."""
    x  = (np.arange(nx) - nx // 2) * dx
    y  = (np.arange(ny) - ny // 2) * dx
    X, Y = np.meshgrid(x, y)
    R2 = X**2 + Y**2 + z**2
    R  = np.sqrt(R2)
    with np.errstate(divide='ignore', invalid='ignore'):
        Kx = C * 3 * X * z / R2**2.5
        Ky = C * 3 * Y * z / R2**2.5
        Kz = C * (3 * z**2 / R2**2.5 - 1 / R2**1.5)
    Kx[R == 0] = Ky[R == 0] = Kz[R == 0] = 0.
    return Kx, Ky, Kz


def field_at_z(spins, z, dx=dx, C=C_dip):
    """Bx, By, Bz at height z via FFT convolution."""
    ny, nx = spins.shape
    Kx, Ky, Kz = dipole_kernels(nx, ny, dx, z, C)
    S = fft2(spins)
    def conv(K): return np.real(ifft2(S * fft2(ifftshift(K))))
    return conv(Kx), conv(Ky), conv(Kz)


Bx0, By0, Bz0 = field_at_z(spins, z_val)
B_max = np.sqrt(Bx0**2 + By0**2 + Bz0**2).max()
print(f"|B|_max  at z = {z_val*1e9:.0f} nm : {B_max*1e3:.3f} mT")
print(f"Bias amplitude : {B_bias_amp*1e3:.1f} mT  (ratio bias/signal ≈ {B_bias_amp/B_max:.1f}×)")

## Four ground-truth variants
Will the reconstruction algorithm works for all of them ?

In [ ]:
# Each variant is (Bx, By, Bz) at z_val
gt_fields = {
    'Classic':              (Bx0,                     By0,                     Bz0                    ),
    'Classic + in-plane Bx': (Bx0 + B_bias_inplane[0], By0 + B_bias_inplane[1], Bz0 + B_bias_inplane[2]),
    #'Classic + Bz bias':    (Bx0 + B_bias_inplane[0],       By0 + B_bias_z[1],       Bz0 + B_bias_z[2]      ),
    'Classic + Bz bias':    (Bx0 + B_bias_z[0],       By0 + B_bias_z[1],       Bz0 + B_bias_z[2]      ),
    'Classic + NV bias':    (Bx0 + B_bias_nv[0],      By0 + B_bias_nv[1],      Bz0 + B_bias_nv[2]     ),
}

print("DC components (spatial mean) for each variant:")
for name, (Bx, By, Bz) in gt_fields.items():
    print(f"  {name:<26}: <Bx>={Bx.mean()*1e3:+.3f} mT  <By>={By.mean()*1e3:+.3f} mT  <Bz>={Bz.mean()*1e3:+.3f} mT")

## Visualise ground-truth fields

In [ ]:
# ── Quiver + bwr helper ──────────────────────────────────────────────────
QSTEP  = 6
xs_q   = np.linspace(0, Lx*1e6, NX)[::QSTEP]
ys_q   = np.linspace(0, Ly*1e6, NY)[::QSTEP]
Xs_q, Ys_q = np.meshgrid(xs_q, ys_q)


def plot_field_bwr(ax, Bx, By, Bz, title, fig=None, vmax_override=None, quiver_ref=None):
    """Bz as bwr colourmap + normalised Bx/By quiver arrows."""
    vmax = vmax_override if vmax_override else max(np.abs(Bz).max() * 1e3, 1e-9)
    im   = ax.imshow(Bz * 1e3, origin='lower', extent=extent_um,
                     cmap='bwr', vmin=-vmax, vmax=vmax, interpolation='nearest', alpha=1)
    U    = Bx[::QSTEP, ::QSTEP]
    V    = By[::QSTEP, ::QSTEP]
    mag  = np.sqrt(U**2 + V**2)
    ref = quiver_ref if quiver_ref is not None else (np.percentile(mag[mag > 0], 90) if (mag > 0).any() else 1.)
    ax.quiver(Xs_q, Ys_q, U / ref, V / ref,
              color='k', alpha=0.99, scale=18, width=0.004, headwidth=3.5)
    ax.set_title(title); ax.set_xlabel('x (μm)'); ax.set_ylabel('y (μm)')
    if fig is not None:
        fig.colorbar(im, ax=ax, label='Bz (mT)', shrink=0.85)
    return im


# ── Figure A: all 4 ground truths ────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5), constrained_layout=True)
fig.suptitle(fr'Ground-truth fields (z $\approx$ {z_val*1e9:.0f} nm)', fontsize=12)
for ax, (name, (Bx, By, Bz)) in zip(axes, gt_fields.items()):
    plot_field_bwr(ax, Bx, By, Bz, name, fig=fig)
plt.savefig(OUT / "simu_groundtruth.pdf")
plt.show()

# ── Figure B: classic field alone ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5.5, 4.5))
Bx, By, Bz = gt_fields['Classic']
plot_field_bwr(ax, Bx, By, Bz, f'Classic field (= with no added bias field, z ≈ {z_val*1e9:.0f} nm)', fig=fig)
plt.tight_layout(); plt.show()

## NV projection $B_\parallel = B\cdot \hat{u}$

In [ ]:
def nv_projection(Bx, By, Bz, nv=nv_hat):
    return Bx * nv[0] + By * nv[1] + Bz * nv[2]

gt_bpar = {name: nv_projection(*B) for name, B in gt_fields.items()}

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5), constrained_layout=True)
fig.suptitle('NV projections B∥  (θ = 125.25°, φ = 90°)', fontsize=12)
for ax, (name, Bpar) in zip(axes, gt_bpar.items()):
    vmax = max(np.abs(Bpar).max() * 1e3, 1e-9)
    im   = ax.imshow(Bpar * 1e3, origin='lower', extent=extent_um,
                     cmap='bwr', vmin=-vmax, vmax=vmax, interpolation='nearest')
    ax.set_title(name, fontsize=9); ax.set_xlabel('x (μm)'); ax.set_ylabel('y (μm)')
    fig.colorbar(im, ax=ax, label='B∥ (mT)', shrink=0.85)
plt.show()

In [ ]:
# ── Simulation projection NV : lattice classique ±1 ─────────────────────

# Lattice de spins classique ±1
spins_classic = generate_spin_lattice(NX, NY, seed=4)

# Champ dipolaire associé
Bx, By, Bz = field_at_z(spins_classic, z_val)

# Projection selon l'axe NV
Bpar = nv_projection(Bx, By, Bz)

# Échelle couleur commune pour B∥
vmax = max(np.abs(Bpar).max() * 1e3, 1e-9)

fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.5), constrained_layout=True)

# Projection NV signée
im0 = axes[0].imshow(
    Bpar * 1e3,
    origin="lower",
    extent=extent_um,
    cmap="bwr",
    vmin=-vmax,
    vmax=vmax,
    interpolation="nearest",
)

axes[0].set_title(r"$B_{\parallel}$", fontsize=15)
axes[0].set_xlabel("x [µm]")
axes[0].set_ylabel("y [µm]")
fig.colorbar(im0, ax=axes[0], label=r"mT", shrink=0.85)

# Valeur absolue de la projection NV
im1 = axes[1].imshow(
    np.abs(Bpar) * 1e3,
    origin="lower",
    extent=extent_um,
    cmap="plasma",
    vmin=0,
    vmax=vmax,
    interpolation="nearest",
)

axes[1].set_title(r"$|B_{\parallel}|$", fontsize=15)
axes[1].set_xlabel("x [µm]")
axes[1].set_ylabel("y [µm]")
fig.colorbar(im1, ax=axes[1], label=r"mT", shrink=0.85)

#fig.suptitle(fr"Simulation of NV projection, z $\approx$ {z_val*1e9:.0f} nm", fontsize=12)

fig.savefig(OUT / "simu_NV_projection_lattice_abs.pdf", bbox_inches="tight", facecolor="white")
plt.show()

## Reconstruction algorithm (Tikhonov in k-space)

**Forward model (k-space):**
$$\hat{B}_\parallel(\mathbf{k}) = D(\mathbf{k})\,\hat{B}_z(\mathbf{k}), \qquad D(\mathbf{k}) = u_z - i\frac{u_x k_x + u_y k_y}{k}$$

**Inversion:**
$$\hat{B}_z = \frac{D^*(\mathbf{k})}{|D|^2 + \lambda(k)}\,\hat{B}_\parallel, \qquad \hat{B}_{x,y} = -i\frac{k_{x,y}}{k}\hat{B}_z$$

**Regularisation:** $\lambda(k) = \lambda_0 \cdot \bigg(1 + (\frac{k}{k_0})^p\bigg)$, $\;k_0 = \alpha/z_{\rm meas}$


**k=0 fix:**
- $\hat{B}_z(0)$ is assigned by convention: $\hat{B}_z(0) = \hat{B}_\parallel(0)/u_z$
- $\hat{B}_x(0)$ and $\hat{B}_y(0)$ are **unrecoverable** from a single projection → forced to 0

In [ ]:
# ── k-space operators (precomputed once) ─────────────────────────────────
kx_arr = 2 * np.pi * np.fft.fftfreq(NX, d=dx)
ky_arr = 2 * np.pi * np.fft.fftfreq(NY, d=dx)
KX, KY = np.meshgrid(kx_arr, ky_arr, indexing='xy')
K      = np.sqrt(KX**2 + KY**2)
K_safe = np.where(K == 0, 1., K)      # avoid 0/0; handled explicitly at k=0

D_op  = uz - 1j * (ux * KX + uy * KY) / K_safe   # forward operator D(k)
D2    = np.abs(D_op)**2



def reconstruct(Bpar_hat, p, alpha_k0, lam0_fac):
    '''
    Tikhonov inversion: B_par -> (Bx, By, Bz).

    Parameters
    ----------
    Bpar_hat   : 2D FFT of the measured projection
    p          : regularisation exponent (steepness of k-space roll-off)
    alpha_k0   : k0 = alpha_k0 / z_val  (physical cut-off scale)
    lam0_fac   : lambda0 = lam0_fac (dimensionless, no median scaling)

    k=0 handling
    ------------
    Bz DC : exact  →  Bz_hat[0,0] = Bpar_hat[0,0] / uz
    Bx/By DC : lost →  set to 0  (in-plane and NV bias DC errors are unavoidable)
    '''
    k0    = alpha_k0 / max(z_val, 1e-12)
    lam_k = lam0_fac * (1.0 + (K / k0) ** p)
    #lam_k = lam0_fac * (1.0 + (K / k0) ** p)

    Bz_hat = np.conj(D_op) * Bpar_hat / (D2 + lam_k)

    # k=0 fix: recover Bz mean; Bx/By means are unknowable
    if abs(uz) > 1e-10:
        Bz_hat[0, 0] = Bpar_hat[0, 0] / uz
    Bx_hat = -1j * (KX / K_safe) * Bz_hat
    By_hat = -1j * (KY / K_safe) * Bz_hat
    Bx_hat[0, 0] = 0.
    By_hat[0, 0] = 0.

    return (np.real(ifft2(Bx_hat)),
            np.real(ifft2(By_hat)),
            np.real(ifft2(Bz_hat)))


def rmse_mT(A, B):
    return float(np.sqrt(np.mean((np.asarray(A) - np.asarray(B))**2))) * 1e3

## Grid search

In [ ]:
# ── Hyperparameter grid ───────────────────────────────────────────────────
p_list        = [1, 2, 4, 7, 10, 15, 20]
alpha_k0_list = [0.5, 1.0, 2.0, 3, 6, 10]
lam0_factors  = np.logspace(-20, 0, 15)

# ── Run grid search for each ground truth ────────────────────────────────
all_results  = {}   # name -> sorted DataFrame
all_best_rec = {}   # name -> (Bx_r, By_r, Bz_r)

for name, (gt_Bx, gt_By, gt_Bz) in gt_fields.items():
    Bpar_hat = fft2(gt_bpar[name])
    rows = []
    for p, ak0, lf in iproduct(p_list, alpha_k0_list, lam0_factors):
        Bx_r, By_r, Bz_r = reconstruct(Bpar_hat, p, ak0, lf)
        ex = rmse_mT(Bx_r, gt_Bx)
        ey = rmse_mT(By_r, gt_By)
        ez = rmse_mT(Bz_r, gt_Bz)
        rows.append({'p': p, 'alpha_k0': ak0, 'lam0_fac': lf,
                     'RMSE_Bx': ex, 'RMSE_By': ey, 'RMSE_Bz': ez,
                     'score': (ex + ey + ez) / 3})

    df = pd.DataFrame(rows).sort_values('score').reset_index(drop=True)
    all_results[name] = df

    best = df.iloc[0]
    all_best_rec[name] = reconstruct(Bpar_hat, int(best.p), best.alpha_k0, best.lam0_fac)

    print(f"\n{'─'*55}")
    print(f"{name}")
    print(f"  Best params : p={int(best.p)},  α·k₀={best.alpha_k0},  λ₀={best.lam0_fac:.2e}")
    print(f"  RMSE        : Bx={best.RMSE_Bx:.3f} mT | By={best.RMSE_By:.3f} mT | Bz={best.RMSE_Bz:.3f} mT")
    print(f"  Mean RMSE   : {best.score:.3f} mT")

## GT vs best reconstruction

In [ ]:
# ── Figure: Ground truth (top) / Reconstruction (bottom) ─────────────────
# Fixed parameters for the manuscript figure; oracle search remains separate.
publication_rec = {name: reconstruct(fft2(gt_bpar[name]), 10, 1.0, 1e-8)
                   for name in gt_fields}
common_vmax = max(np.max(np.abs(b[2])) * 1e3
                  for b in list(gt_fields.values()) + list(publication_rec.values()))
common_quiver_ref = max(np.percentile(np.concatenate([
    np.hypot(b[0][::QSTEP, ::QSTEP], b[1][::QSTEP, ::QSTEP]).ravel()
    for b in list(gt_fields.values()) + list(publication_rec.values())]), 90), 1e-12)
fig, axes = plt.subplots(2, 4, figsize=(20, 9), constrained_layout=True)
fig.suptitle(fr'Ground Truth vs Reconstructed fields (z $\approx$ {z_val*1e9:.0f} nm)', fontsize=12)

ims_top = []
ims_bottom = []

for j, (name, (Bx_gt, By_gt, Bz_gt)) in enumerate(gt_fields.items()):
    im = plot_field_bwr(axes[0, j], Bx_gt, By_gt, Bz_gt, f'{name} (GT)', vmax_override=common_vmax, quiver_ref=common_quiver_ref)
    ims_top.append(im)

    Bx_r, By_r, Bz_r = publication_rec[name]
    im = plot_field_bwr(axes[1, j], Bx_r, By_r, Bz_r, f'{name} (Reconstructed)', vmax_override=common_vmax, quiver_ref=common_quiver_ref)
    ims_bottom.append(im)

fig.colorbar(ims_top[0], ax=axes[0, :], label='Bz (mT)', shrink=0.85)
fig.colorbar(ims_bottom[0], ax=axes[1, :], label='Bz (mT)', shrink=0.85)

plt.savefig(OUT / "gt_vs_reconstructed_2x4.pdf")
plt.show()

In [ ]:
# ── New final visualisation: GT / Reconstruction / Difference in quiver style ──
from matplotlib.patches import Patch

# ── New final visualisation: GT / Reconstruction / Difference in quiver style ──

def plot_field_quiver(ax, Bx, By, Bz, title, ref_arrow, vmax_mT, cmap='bwr'):
    im = ax.imshow(Bz * 1e3, origin='lower', extent=extent_um,
                   cmap=cmap, vmin=-vmax_mT, vmax=vmax_mT, interpolation='nearest')
    U = Bx[::QSTEP, ::QSTEP]
    V = By[::QSTEP, ::QSTEP]
    ax.quiver(Xs_q, Ys_q, U / ref_arrow, V / ref_arrow,
              color='k', alpha=0.95, scale=18, width=0.004, headwidth=3.5)
    ax.set_title(title)
    ax.set_xlabel('x (μm)')
    ax.set_ylabel('y (μm)')
    return im


for name, (gt_Bx, gt_By, gt_Bz) in gt_fields.items():
    Bx_r, By_r, Bz_r = all_best_rec[name]

    dBx = Bx_r - gt_Bx
    dBy = By_r - gt_By
    dBz = Bz_r - gt_Bz

    # Same arrow scale for GT and reconstruction
    mag_gt = np.sqrt(gt_Bx[::QSTEP, ::QSTEP]**2 + gt_By[::QSTEP, ::QSTEP]**2)
    ref_main = np.percentile(mag_gt[mag_gt > 0], 90) if (mag_gt > 0).any() else 1.0
    ref_main = max(ref_main, 1e-12)

    # Separate arrow scale for error field so the difference remains visible
    mag_diff = np.sqrt(dBx[::QSTEP, ::QSTEP]**2 + dBy[::QSTEP, ::QSTEP]**2)
    ref_diff = np.percentile(mag_diff[mag_diff > 0], 90) if (mag_diff > 0).any() else ref_main
    ref_diff = max(ref_diff, 1e-12)

    # Ratio of arrow magnification in the error panel
    arrow_zoom = ref_main / ref_diff

    # Same color scale for GT and reconstruction, separate one for difference
    vmax_main = max(np.abs(gt_Bz).max(), np.abs(Bz_r).max()) * 1e3
    vmax_main = max(vmax_main, 1e-9)
    vmax_diff = max(np.abs(dBz).max() * 1e3, 1e-9)

    rmse_bx = rmse_mT(Bx_r, gt_Bx)
    rmse_by = rmse_mT(By_r, gt_By)
    rmse_bz = rmse_mT(Bz_r, gt_Bz)

    fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.8), constrained_layout=True)
    fig.suptitle(
        f"{name}  —  GT / Reconstruction / Error"
        f"\nRMSE: Bx={rmse_bx:.3f} mT, By={rmse_by:.3f} mT, Bz={rmse_bz:.3f} mT",
        fontsize=12
    )

    im0 = plot_field_quiver(
        axes[0], gt_Bx, gt_By, gt_Bz,
        title='Ground truth',
        ref_arrow=ref_main,
        vmax_mT=vmax_main
    )

    im1 = plot_field_quiver(
        axes[1], Bx_r, By_r, Bz_r,
        title='Reconstructed',
        ref_arrow=ref_main,
        vmax_mT=vmax_main
    )

    im2 = plot_field_quiver(
        axes[2], dBx, dBy, dBz,
        title='Difference (recon - GT)',
        ref_arrow=ref_diff,
        vmax_mT=vmax_diff
    )

    # Legend with magnification ratio for the error arrows
    axes[2].legend(
        handles=[Patch(facecolor='none', edgecolor='none',
                       label=f'Arrow magnification: ×{arrow_zoom:.1f} vs GT/Reconstructed')],
        loc='upper left',
        framealpha=0.9,
        handlelength=0,
        handletextpad=0
    )

    cbar_main = fig.colorbar(im0, ax=axes[:2], shrink=0.88)
    cbar_main.set_label('Bz (mT)')

    cbar_diff = fig.colorbar(im2, ax=axes[2], shrink=0.88)
    cbar_diff.set_label('ΔBz (mT)')

    plt.show()

## Summary

In [ ]:
rows = []
for name, df in all_results.items():
    best = df.iloc[0]
    rows.append({
        'Ground truth':    name,
        'p':               int(best.p),
        'α·k₀':           best.alpha_k0,
        'λ₀ factor':      f"{best.lam0_fac:.1e}",
        'RMSE Bx (mT)':   f"{best.RMSE_Bx:.3f}",
        'RMSE By (mT)':   f"{best.RMSE_By:.3f}",
        'RMSE Bz (mT)':   f"{best.RMSE_Bz:.3f}",
        'Mean RMSE (mT)':  f"{best.score:.3f}",
    })

df_summary = pd.DataFrame(rows)
print("\n=== Reconstruction quality summary ===")
print(df_summary.to_string(index=False))

# ── Note on k=0 limitation ────────────────────────────────────────────────
print("\n--- k=0 note ---")
print(f"  NV axis: ux={ux:.4f}, uy={uy:.4f}, uz={uz:.4f}")
print(f"  Bz DC  : assigned by convention  (uz={uz:.4f} ≠ 0)")
print(f"  Bx DC  : NOT recoverable (systematic offset for 'in-plane' and 'NV' bias variants)")
print(f"  By DC  : NOT recoverable (idem)")

### Fixed protocol and interpretation

The oracle grid search selects parameters using the ground truth of each evaluated map.
Its Gaussian noise (3 mT, one realization per case) is an illustrative stress test,
not an experimental noise model or an uncertainty estimate. Averaged best parameters
do not define a separately validated configuration. The fixed reference protocol is
lambda0=1e-8, p=10, alpha=1, with no median scaling. The DC convention in these
diagnostic benchmarks assigns the mean projection to Bz; compare AC fields when
evaluating spatial reconstruction. The sign-recovery and experimental notebooks use
the same spectral filter and zero reconstructed means.


In [ ]:
# Evaluate fixed parameters independently of the oracle parameter selection.
from sim_odmr_field_helpers import make_error_dataframe
fixed_rows = []
if 'field_cases' in globals():
    cases_fixed = {name: (case['Bx'], case['By'], case['Bz'])
                   for name, case in field_cases.items()}
else:
    cases_fixed = gt_fields
for name, truth in cases_fixed.items():
    projection = nv_projection(*truth)
    reconstruction = reconstruct(fft2(projection), 10, 1.0, 1e-8)
    metrics = make_error_dataframe(projection*1e3, projection*1e3, reconstruction, truth)
    metrics.insert(0, 'case', name)
    fixed_rows.append(metrics)
fixed_metrics = pd.concat(fixed_rows, ignore_index=True)
display(fixed_metrics)


In [ ]:
fixed_metrics.to_csv(OUT / "fixed_protocol_metrics.csv", index=False)
df_summary.to_csv(OUT / "oracle_summary.csv", index=False)
